# PyTorch 张量变换与重塑教程

> 本教程整理自 `LLM-Interview-Code` 项目中实际使用的张量操作

在 LLM 开发中，张量形状变换是极其重要的技能。无论是多头注意力的分头操作、位置编码的维度扩展，还是 MoE 的专家路由，都需要熟练掌握这些操作。

## 本教程覆盖的操作

| 操作 | 用途 | 项目应用 |
|------|------|----------|
| `view()` | 张量重塑 | 多头注意力分头 |
| `reshape()` | 张量重塑（可复制内存） | GQA中KV扩展 |
| `transpose()` | 维度交换 | 注意力计算QK^T |
| `contiguous()` | 内存连续化 | 注意力输出重塑 |
| `squeeze()`/`unsqueeze()` | 维度增减 | RoPE维度扩展 |
| `expand()` | 广播式扩展 | GQA的KV头重复 |
| `cat()` | 张量拼接 | RoPE角度拼接 |
| `split()`/`chunk()` | 张量分割 | MLA分割K/V |
| `where()` | 条件选择 | MoE专家路由 |

In [1]:
import torch
print(f"PyTorch 版本: {torch.__version__}")

PyTorch 版本: 2.5.1+cu121


---
## 1. `view()` - 张量重塑

`view()` 是最常用的张量重塑操作，它返回一个**共享内存**的新视图。

### 语法
```python
tensor.view(*shape)  # shape 中可以有最多一个 -1，表示自动推断
```

### 关键点
- **共享内存**：修改 view 后的张量会影响原张量
- **要求连续**：张量必须在内存中连续才能使用 view
- **元素总数不变**：重塑前后元素数量必须相同

In [2]:
# 基础示例
x = torch.arange(12)  # [12]
print(f"原始张量: {x.shape}")

# 重塑为 3x4
y = x.view(3, 4)
print(f"view(3, 4): {y.shape}")

# 使用 -1 自动推断
z = x.view(-1, 4)  # 自动推断第一个维度为 3
print(f"view(-1, 4): {z.shape}")

# 共享内存验证
y[0, 0] = 999
print(f"修改 y 后，x[0] = {x[0]}")  # 原张量也被修改

原始张量: torch.Size([12])
view(3, 4): torch.Size([3, 4])
view(-1, 4): torch.Size([3, 4])
修改 y 后，x[0] = 999


### 项目应用：多头注意力中的分头操作

来源：`attention/MultiHeadAttention.py:37`

```python
# 线性变换并分头
# q: (B, S, D) -> (B, S, H, head_dim) -> (B, H, S, head_dim)
q = q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
```

In [3]:
# 模拟多头注意力的分头操作
batch_size, seq_len, model_dim, num_heads = 2, 10, 64, 8
head_dim = model_dim // num_heads  # 8

# 模拟线性层输出
q = torch.randn(batch_size, seq_len, model_dim)
print(f"线性层输出 q: {q.shape}")

# 分头：将 model_dim 拆分为 num_heads × head_dim
q = q.view(batch_size, seq_len, num_heads, head_dim)
print(f"分头后 q: {q.shape}")

# 转置：将 num_heads 维度移到前面，便于并行计算
q = q.transpose(1, 2)
print(f"转置后 q: {q.shape}  # (B, H, S, head_dim)")

线性层输出 q: torch.Size([2, 10, 64])
分头后 q: torch.Size([2, 10, 8, 8])
转置后 q: torch.Size([2, 8, 10, 8])  # (B, H, S, head_dim)


---
## 2. `reshape()` vs `view()`

`reshape()` 与 `view()` 功能类似，但有关键区别：

| 操作 | 内存行为 | 使用场景 |
|------|----------|----------|
| `view()` | 必须共享内存，张量必须连续 | 确定张量连续时 |
| `reshape()` | 尽量共享内存，必要时复制 | 不确定张量是否连续时 |

### 最佳实践
- 如果确定张量连续，优先用 `view()`（更明确意图）
- 如果不确定，用 `reshape()`（更安全）

In [4]:
# view() 对非连续张量会报错
x = torch.arange(12).view(3, 4)
y = x.t()  # 转置后内存不连续
print(f"转置后是否连续: {y.is_contiguous()}")

try:
    z = y.view(4, 3)  # 会报错
except RuntimeError as e:
    print(f"view() 报错: {e}")

# reshape() 会自动处理
z = y.reshape(4, 3)
print(f"reshape() 成功: {z.shape}")

转置后是否连续: False
reshape() 成功: torch.Size([4, 3])


### 项目应用：GQA 中的 KV 扩展

来源：`attention/GroupQueryAttention.py:44`

```python
# 展平 num_kv_heads 和 n_rep: [Batch, num_kv_heads * n_rep, seq_len, head_dim]
x = x.reshape(batch, num_kv_heads * n_rep, seq_len, head_dims)
```

In [5]:
# 模拟 GQA 中的 repeat_kv 操作
batch, num_kv_heads, n_rep, seq_len, head_dim = 2, 4, 2, 10, 16

# 原始 KV: [batch, num_kv_heads, seq_len, head_dim]
k = torch.randn(batch, num_kv_heads, seq_len, head_dim)
print(f"原始 K: {k.shape}")

# 1. 新增维度: [batch, num_kv_heads, 1, seq_len, head_dim]
k = k[:, :, None, :, :]
print(f"unsqueeze 后: {k.shape}")

# 2. 扩展: [batch, num_kv_heads, n_rep, seq_len, head_dim]
k = k.expand(batch, num_kv_heads, n_rep, seq_len, head_dim)
print(f"expand 后: {k.shape}")

# 3. reshape 展平: [batch, num_kv_heads * n_rep, seq_len, head_dim]
k = k.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)
print(f"reshape 后: {k.shape}  # num_heads = {num_kv_heads * n_rep}")

原始 K: torch.Size([2, 4, 10, 16])
unsqueeze 后: torch.Size([2, 4, 1, 10, 16])
expand 后: torch.Size([2, 4, 2, 10, 16])
reshape 后: torch.Size([2, 8, 10, 16])  # num_heads = 8


---
## 3. `transpose()` 与 `permute()` - 维度交换

### `transpose()` - 交换两个维度
```python
tensor.transpose(dim0, dim1)  # 只能交换两个维度
```

### `permute()` - 重排所有维度
```python
tensor.permute(*dims)  # 可以重排任意数量的维度
```

### 关键点
- 转置后张量**不连续**，如果后续需要 `view()`，必须先 `contiguous()`

In [6]:
# transpose() 示例
x = torch.randn(2, 3, 4)
print(f"原始: {x.shape}")

y = x.transpose(0, 1)  # 交换第0维和第1维
print(f"transpose(0, 1): {y.shape}")

# permute() 示例 - 可以重排所有维度
z = x.permute(2, 0, 1)  # 将维度重排为 (4, 2, 3)
print(f"permute(2, 0, 1): {z.shape}")

原始: torch.Size([2, 3, 4])
transpose(0, 1): torch.Size([3, 2, 4])
permute(2, 0, 1): torch.Size([4, 2, 3])


### 项目应用：注意力计算中的 QK^T

来源：`attention/MultiHeadAttention.py:42`

```python
# scores: [B, H, S, S] = [B, H, S, D] @ [B, H, D, S]
scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
```

In [7]:
import math

# 模拟注意力分数计算
batch, num_heads, seq_len, head_dim = 2, 8, 10, 16

q = torch.randn(batch, num_heads, seq_len, head_dim)
k = torch.randn(batch, num_heads, seq_len, head_dim)

print(f"Q shape: {q.shape}")
print(f"K shape: {k.shape}")
print(f"K.transpose(-2, -1) shape: {k.transpose(-2, -1).shape}")

# Q @ K^T: [B, H, S, D] @ [B, H, D, S] -> [B, H, S, S]
scores = torch.matmul(q, k.transpose(-2, -1))
print(f"\n注意力分数 scores: {scores.shape}")

# 注意：转置后张量不连续
k_t = k.transpose(-2, -1)
print(f"转置后是否连续: {k_t.is_contiguous()}")

Q shape: torch.Size([2, 8, 10, 16])
K shape: torch.Size([2, 8, 10, 16])
K.transpose(-2, -1) shape: torch.Size([2, 8, 16, 10])

注意力分数 scores: torch.Size([2, 8, 10, 10])
转置后是否连续: False


---
## 4. `contiguous()` - 内存连续化

转置、permute 等操作后，张量在内存中不再连续。如果要使用 `view()`，必须先调用 `contiguous()`。

### 语法
```python
tensor.contiguous()  # 返回内存连续的副本（如果原来不连续）
```

### 为什么需要连续？
- `view()` 要求张量在内存中连续存储
- 转置操作只是改变了步长（stride），没有移动数据

In [8]:
# contiguous() 的作用
x = torch.arange(12).view(3, 4)
y = x.transpose(0, 1)

print(f"转置后是否连续: {y.is_contiguous()}")

# 尝试 view() 会报错
try:
    y.view(-1)
except RuntimeError as e:
    print(f"view() 报错: {e}")

# 先 contiguous() 再 view()
z = y.contiguous().view(-1)
print(f"contiguous().view(-1) 成功: {z.shape}")

转置后是否连续: False
view() 报错: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
contiguous().view(-1) 成功: torch.Size([12])


### 项目应用：注意力输出重塑

来源：`attention/MultiHeadAttention.py:50-53`

```python
# context: (B, H, S, head_dim) -> (B, S, H, head_dim)
context = context.transpose(1, 2)
context = context.contiguous()
# output: (B, S, D)
output = context.view(batch_size, -1, self.model_dim)
```

In [9]:
# 模拟注意力输出重塑
batch, num_heads, seq_len, head_dim, model_dim = 2, 8, 10, 8, 64

# 注意力输出: [B, H, S, head_dim]
context = torch.randn(batch, num_heads, seq_len, head_dim)
print(f"注意力输出 context: {context.shape}")
print(f"是否连续: {context.is_contiguous()}")

# 1. 转置: [B, S, H, head_dim]
context = context.transpose(1, 2)
print(f"\n转置后 context: {context.shape}")
print(f"是否连续: {context.is_contiguous()}")

# 2. 连续化
context = context.contiguous()
print(f"\ncontiguous 后是否连续: {context.is_contiguous()}")

# 3. 合并多头: [B, S, D]
output = context.view(batch, seq_len, model_dim)
print(f"\n最终输出 output: {output.shape}")

注意力输出 context: torch.Size([2, 8, 10, 8])
是否连续: True

转置后 context: torch.Size([2, 10, 8, 8])
是否连续: False

contiguous 后是否连续: True

最终输出 output: torch.Size([2, 10, 64])


---
## 5. `squeeze()` 与 `unsqueeze()` - 维度增减

### `squeeze()` - 删除大小为 1 的维度
```python
tensor.squeeze()       # 删除所有大小为1的维度
tensor.squeeze(dim)    # 删除指定维度（如果大小为1）
```

### `unsqueeze()` - 在指定位置插入大小为 1 的维度
```python
tensor.unsqueeze(dim)  # 在 dim 位置插入新维度
```

### 常见用途
- `unsqueeze()`: 扩展维度以便广播
- `squeeze()`: 去除多余维度

In [10]:
# unsqueeze() 示例
x = torch.randn(3, 4)
print(f"原始: {x.shape}")

y = x.unsqueeze(0)  # 在第0维插入
print(f"unsqueeze(0): {y.shape}")

z = x.unsqueeze(-1)  # 在最后一维插入
print(f"unsqueeze(-1): {z.shape}")

# squeeze() 示例
a = torch.randn(1, 3, 1, 4)
print(f"\n原始: {a.shape}")
print(f"squeeze(): {a.squeeze().shape}")  # 删除所有大小为1的维度
print(f"squeeze(0): {a.squeeze(0).shape}")  # 只删除第0维

原始: torch.Size([3, 4])
unsqueeze(0): torch.Size([1, 3, 4])
unsqueeze(-1): torch.Size([3, 4, 1])

原始: torch.Size([1, 3, 1, 4])
squeeze(): torch.Size([3, 4])
squeeze(0): torch.Size([3, 1, 4])


### 项目应用：RoPE 中的维度扩展

来源：`position/RotaryEmbedding.py:31-32`

```python
cos = self.cos[:seq_len].view(1, seq_len, 1, self.dim)  # (1, seq_len, 1, dim)
sin = self.sin[:seq_len].view(1, seq_len, 1, self.dim)  # (1, seq_len, 1, dim)
```

这里的 `view` 操作相当于 `unsqueeze`，目的是让 cos/sin 能够与 [B, S, H, D] 的 Q、K 进行广播。

In [11]:
# 模拟 RoPE 中的维度扩展
seq_len, dim = 10, 16

# 预计算的 cos/sin: [max_seq_len, dim]
cos = torch.randn(2048, dim)
sin = torch.randn(2048, dim)

# 取当前序列长度并扩展维度用于广播
cos = cos[:seq_len].view(1, seq_len, 1, dim)  # 等价于 unsqueeze
sin = sin[:seq_len].view(1, seq_len, 1, dim)
print(f"cos: {cos.shape}")
print(f"sin: {sin.shape}")

# 与 Q、K 进行广播
batch, num_heads = 2, 8
q = torch.randn(batch, seq_len, num_heads, dim)

# 广播: [1, S, 1, D] * [B, S, H, D] -> [B, S, H, D]
q_rotated = q * cos
print(f"\nQ rotated: {q_rotated.shape}")

cos: torch.Size([1, 10, 1, 16])
sin: torch.Size([1, 10, 1, 16])

Q rotated: torch.Size([2, 10, 8, 16])


---
## 6. `expand()` - 广播式扩展

`expand()` 将张量广播到更大形状，**不复制内存**。

### 语法
```python
tensor.expand(*size)  # 将大小为1的维度扩展到指定大小
```

### 关键点
- 只能扩展大小为 1 的维度
- **不复制内存**，只是创建新视图
- 如果需要复制内存，使用 `repeat()`

In [12]:
# expand() 示例
x = torch.randn(1, 3, 1, 4)
print(f"原始: {x.shape}")

y = x.expand(2, 3, 5, 4)  # 扩展第0维和第2维
print(f"expand(2, 3, 5, 4): {y.shape}")

# expand 不复制内存
print(f"\n内存共享验证:")
x[0, 0, 0, 0] = 999
print(f"修改 x 后，y[0, 0, 0, 0] = {y[0, 0, 0, 0]}")
print(f"y[1, 0, 0, 0] = {y[1, 0, 0, 0]}  # 扩展的维度也共享相同值")

原始: torch.Size([1, 3, 1, 4])
expand(2, 3, 5, 4): torch.Size([2, 3, 5, 4])

内存共享验证:
修改 x 后，y[0, 0, 0, 0] = 999.0
y[1, 0, 0, 0] = 999.0  # 扩展的维度也共享相同值


### `expand()` vs `repeat()`

| 操作 | 内存行为 | 适用场景 |
|------|----------|----------|
| `expand()` | 不复制内存 | 只读操作、广播计算 |
| `repeat()` | 复制内存 | 需要修改副本、需要独立内存 |

In [13]:
# repeat() 示例 - 实际复制内存
x = torch.tensor([[1, 2]])
print(f"原始: {x.shape}")

y = x.repeat(2, 3)  # 在第0维重复2次，第1维重复3次
print(f"repeat(2, 3): \n{y}")
print(f"shape: {y.shape}")

# repeat 后修改不影响原张量
y[0, 0] = 999
print(f"\n修改 y 后，x = {x}")  # x 不受影响

原始: torch.Size([1, 2])
repeat(2, 3): 
tensor([[1, 2, 1, 2, 1, 2],
        [1, 2, 1, 2, 1, 2]])
shape: torch.Size([2, 6])

修改 y 后，x = tensor([[1, 2]])


### 项目应用：GQA 中的 KV 头重复

来源：`attention/GroupQueryAttention.py:38-41`

```python
# 1. 新增维度: [Batch, num_kv_heads, 1, seq_len, head_dim]
x = x[:, :, None, :, :]

# 2. 扩展: [Batch, num_kv_heads, n_rep, seq_len, head_dim]
x = x.expand(batch, num_kv_heads, n_rep, seq_len, head_dims)
```

In [14]:
# 模拟 GQA 的 repeat_kv 操作
batch, num_kv_heads, seq_len, head_dim, n_rep = 2, 4, 10, 16, 2

# KV 张量
k = torch.randn(batch, num_kv_heads, seq_len, head_dim)
print(f"原始 K: {k.shape}")

# 步骤1: unsqueeze
k = k[:, :, None, :, :]
print(f"unsqueeze 后: {k.shape}")

# 步骤2: expand
k = k.expand(batch, num_kv_heads, n_rep, seq_len, head_dim)
print(f"expand 后: {k.shape}")

# 步骤3: reshape
k = k.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)
print(f"reshape 后: {k.shape}  # num_heads = {num_kv_heads * n_rep}")

原始 K: torch.Size([2, 4, 10, 16])
unsqueeze 后: torch.Size([2, 4, 1, 10, 16])
expand 后: torch.Size([2, 4, 2, 10, 16])
reshape 后: torch.Size([2, 8, 10, 16])  # num_heads = 8


---
## 7. `cat()` 与 `stack()` - 张量拼接

### `cat()` (concatenate) - 在已有维度拼接
```python
torch.cat([tensor1, tensor2, ...], dim)  # 在指定维度拼接
```

### `stack()` - 创建新维度堆叠
```python
torch.stack([tensor1, tensor2, ...], dim)  # 创建新维度堆叠
```

### 区别
- `cat()`: 不增加维度，在已有维度上拼接
- `stack()`: 增加一个新维度

In [15]:
# cat() 示例
a = torch.randn(2, 3)
b = torch.randn(2, 3)

print(f"a: {a.shape}, b: {b.shape}")

# 在第0维拼接
c = torch.cat([a, b], dim=0)
print(f"cat(dim=0): {c.shape}")

# 在第1维拼接
d = torch.cat([a, b], dim=1)
print(f"cat(dim=1): {d.shape}")

# stack() 示例
e = torch.stack([a, b], dim=0)  # 创建新维度
print(f"\nstack(dim=0): {e.shape}")

a: torch.Size([2, 3]), b: torch.Size([2, 3])
cat(dim=0): torch.Size([4, 3])
cat(dim=1): torch.Size([2, 6])

stack(dim=0): torch.Size([2, 2, 3])


### 项目应用：RoPE 中的角度拼接

来源：`position/RotaryEmbedding.py:25` 和 `35-36`

```python
# 预计算时：拼接角度
angles = torch.cat((angles, angles), dim=-1)  # [S, d/2] -> [S, d]

# rotate_half 中：拼接旋转后的两半
x1, x2 = torch.chunk(x, 2, dim=-1)
return torch.cat((-x2, x1), dim=-1)
```

In [16]:
# 模拟 RoPE 的 rotate_half 操作
x = torch.randn(2, 10, 8, 16)  # [B, S, H, D]
print(f"输入 x: {x.shape}")

# 分成两半
x1, x2 = torch.chunk(x, 2, dim=-1)
print(f"\nx1: {x1.shape}")
print(f"x2: {x2.shape}")

# 旋转并拼接: [-x2, x1]
rotated = torch.cat((-x2, x1), dim=-1)
print(f"\nrotated: {rotated.shape}")

# 验证：前半部分是 -x2，后半部分是 x1
print(f"\n验证: rotated[..., :8] 等于 -x2: {torch.allclose(rotated[..., :8], -x2)}")
print(f"验证: rotated[..., 8:] 等于 x1: {torch.allclose(rotated[..., 8:], x1)}")

输入 x: torch.Size([2, 10, 8, 16])

x1: torch.Size([2, 10, 8, 8])
x2: torch.Size([2, 10, 8, 8])

rotated: torch.Size([2, 10, 8, 16])

验证: rotated[..., :8] 等于 -x2: True
验证: rotated[..., 8:] 等于 x1: True


---
## 8. `split()` 与 `chunk()` - 张量分割

### `split()` - 按指定大小分割
```python
torch.split(tensor, split_size_or_sections, dim)
# split_size_or_sections 可以是整数或列表
```

### `chunk()` - 按指定份数分割
```python
torch.chunk(tensor, chunks, dim)  # 分成 chunks 份
```

### 区别
- `split()`: 指定每份的大小
- `chunk()`: 指定分割的份数

In [17]:
# split() 示例
x = torch.arange(10)
print(f"原始: {x.shape}")

# 按大小分割（返回一个 tuple）
result = torch.split(x, 3, dim=0)  # 每份3个，最后一份可能不足
print(f"split(3): 返回 {len(result)} 个张量")
for i, t in enumerate(result):
    print(f"  第{i}份: {t.tolist()}")

# 按列表指定大小
a, b = torch.split(x, [2, 8], dim=0)
print(f"\nsplit([2, 8]): a={a.tolist()}, b={b.tolist()}")

# chunk() 示例
x = torch.arange(12).view(3, 4)
print(f"\n原始: {x.shape}")

a, b = torch.chunk(x, 2, dim=1)  # 分成2份
print(f"chunk(2, dim=1): a={a.shape}, b={b.shape}")
print(f"  a=\n{a}")
print(f"  b=\n{b}")

原始: torch.Size([10])
split(3): 返回 4 个张量
  第0份: [0, 1, 2]
  第1份: [3, 4, 5]
  第2份: [6, 7, 8]
  第3份: [9]

split([2, 8]): a=[0, 1], b=[2, 3, 4, 5, 6, 7, 8, 9]

原始: torch.Size([3, 4])
chunk(2, dim=1): a=torch.Size([3, 2]), b=torch.Size([3, 2])
  a=
tensor([[0, 1],
        [4, 5],
        [8, 9]])
  b=
tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])


### 项目应用：MLA 中分割 K/V

来源：`attention/MultiLatentAttention.py:41`

```python
# split content and rope
# k_content: (batch_size, seq_len, num_heads, head_dim)
# k_rope: (batch_size, seq_len, num_heads, rope_dim)
# v_content: (batch_size, seq_len, num_heads, head_dim)
k_content, k_rope, v_content = torch.split(
    kv_full, [self.head_dim, self.rope_dim, self.head_dim], dim=-1
)
```

In [18]:
# 模拟 MLA 的 KV 分割
batch, seq_len, num_heads, head_dim, rope_dim = 2, 10, 8, 16, 4

# kv_full: [B, S, H, head_dim + rope_dim + head_dim]
kv_full = torch.randn(batch, seq_len, num_heads, head_dim + rope_dim + head_dim)
print(f"kv_full: {kv_full.shape}")

# 按指定大小分割
k_content, k_rope, v_content = torch.split(
    kv_full, [head_dim, rope_dim, head_dim], dim=-1
)

print(f"\nk_content: {k_content.shape}")
print(f"k_rope: {k_rope.shape}")
print(f"v_content: {v_content.shape}")

kv_full: torch.Size([2, 10, 8, 36])

k_content: torch.Size([2, 10, 8, 16])
k_rope: torch.Size([2, 10, 8, 4])
v_content: torch.Size([2, 10, 8, 16])


---
## 9. `where()` - 条件选择

### 语法
```python
torch.where(condition, x, y)  # 条件为真选 x，否则选 y
torch.where(condition)        # 返回满足条件的索引
```

### 两种用法
1. 三元选择：`where(cond, x, y)` 相当于 `cond ? x : y`
2. 索引查找：`where(cond)` 返回满足条件的索引元组

In [19]:
# 用法1：三元选择
x = torch.tensor([1, 2, 3, 4, 5])
y = torch.tensor([10, 20, 30, 40, 50])
cond = x > 3

result = torch.where(cond, x, y)
print(f"cond: {cond.tolist()}")
print(f"where(cond, x, y): {result.tolist()}")

# 用法2：索引查找
mask = torch.tensor([[True, False, True],
                     [False, True, False]])
indices = torch.where(mask)
print(f"\nmask:\n{mask}")
print(f"where(mask): row={indices[0].tolist()}, col={indices[1].tolist()}")

cond: [False, False, False, True, True]
where(cond, x, y): [10, 20, 30, 4, 5]

mask:
tensor([[ True, False,  True],
        [False,  True, False]])
where(mask): row=[0, 0, 1], col=[0, 2, 1]


### 项目应用：MoE 中找出选中专家的 token

来源：`ffn/MoE.py:46-47`

```python
# 找出选择了第i个专家的 token
mask = (indices == i)  # bool: [batch * seq_len, top_k]
token_indices, top_k_pos = torch.where(mask)
```

这里 `torch.where(mask)` 返回满足条件的索引，用于定位哪些 token 选择了当前专家。

In [20]:
# 模拟 MoE 的专家路由
num_tokens, num_experts, top_k = 10, 8, 2

# 每个 token 选中的 top-k 专家索引
indices = torch.randint(0, num_experts, (num_tokens, top_k))
print(f"indices: {indices.shape}")
print(f"indices:\n{indices}")

# 找出选择了专家 3 的 token
expert_id = 3
mask = (indices == expert_id)
print(f"\nmask (indices == 3):\n{mask.int()}")

# 获取索引
token_indices, top_k_pos = torch.where(mask)
print(f"\n选中专家3的 token 索引: {token_indices.tolist()}")
print(f"在 top-k 中的位置: {top_k_pos.tolist()}")

# 使用这些索引提取对应的 token
x = torch.randn(num_tokens, 64)  # 假设 model_dim=64
selected_tokens = x[token_indices]
print(f"\n选中 token 的特征: {selected_tokens.shape}")

indices: torch.Size([10, 2])
indices:
tensor([[5, 0],
        [5, 3],
        [2, 4],
        [6, 6],
        [2, 0],
        [6, 2],
        [2, 7],
        [5, 2],
        [3, 4],
        [4, 5]])

mask (indices == 3):
tensor([[0, 0],
        [0, 1],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [1, 0],
        [0, 0]], dtype=torch.int32)

选中专家3的 token 索引: [1, 8]
在 top-k 中的位置: [1, 0]

选中 token 的特征: torch.Size([2, 64])


---
## 综合案例：实现多头注意力

下面使用本项目中的实际代码，展示所有操作的完整应用。

In [21]:
import math

def multi_head_attention(q, k, v, num_heads):
    """
    简化版多头注意力，展示张量操作

    输入: q, k, v: [B, S, D]
    输出: [B, S, D]
    """
    batch_size, seq_len, model_dim = q.shape
    head_dim = model_dim // num_heads

    print(f"输入: {q.shape}")

    # 1. view() + transpose(): 分头
    q = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    k = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    v = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    print(f"分头后: {q.shape}")  # [B, H, S, head_dim]

    # 2. transpose(): 计算注意力分数
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
    print(f"注意力分数: {scores.shape}")  # [B, H, S, S]

    # 3. softmax
    attn_weights = torch.softmax(scores, dim=-1)

    # 4. 加权求和
    context = torch.matmul(attn_weights, v)
    print(f"context: {context.shape}")  # [B, H, S, head_dim]

    # 5. transpose() + contiguous() + view(): 合并多头
    context = context.transpose(1, 2).contiguous()
    output = context.view(batch_size, seq_len, model_dim)
    print(f"输出: {output.shape}")  # [B, S, D]

    return output

# 测试
B, S, D, H = 2, 10, 64, 8
q = torch.randn(B, S, D)
k = torch.randn(B, S, D)
v = torch.randn(B, S, D)

output = multi_head_attention(q, k, v, H)

输入: torch.Size([2, 10, 64])
分头后: torch.Size([2, 8, 10, 8])
注意力分数: torch.Size([2, 8, 10, 10])
context: torch.Size([2, 8, 10, 8])
输出: torch.Size([2, 10, 64])


---
## 总结

| 操作 | 作用 | 是否复制内存 | 常见用途 |
|------|------|--------------|----------|
| `view()` | 重塑形状 | 否 | 分头、展平 |
| `reshape()` | 重塑形状 | 必要时是 | 安全重塑 |
| `transpose()` | 交换两维 | 否 | 注意力计算 |
| `permute()` | 重排所有维 | 否 | 复杂维度重排 |
| `contiguous()` | 内存连续化 | 必要时是 | view前确保连续 |
| `unsqueeze()` | 插入维度 | 否 | 广播准备 |
| `squeeze()` | 删除维度 | 否 | 去除冗余维 |
| `expand()` | 扩展维度 | 否 | 只读广播 |
| `repeat()` | 复制扩展 | 是 | 需要独立副本 |
| `cat()` | 拼接 | 视情况 | 合并张量 |
| `stack()` | 堆叠 | 视情况 | 增维合并 |
| `split()` | 按大小分割 | 否 | 分离特征 |
| `chunk()` | 按份数分割 | 否 | 均分张量 |
| `where()` | 条件选择 | 视情况 | 掩码、索引 |

### 最佳实践
1. 优先使用 `view()` 而非 `reshape()`（如果确定张量连续）
2. `transpose()` 后如需 `view()`，必须先 `contiguous()`
3. 只读扩展用 `expand()`，需要副本用 `repeat()`
4. 使用 `-1` 让 PyTorch 自动推断维度大小